In [1]:
from sentence_transformers import SentenceTransformer, CrossEncoder

c:\Users\asus\Desktop\genai-practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!pip install -q langchain-community langchain-huggingface faiss-cpu sentence-transformers

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

documents = [
    "Our subscription plans include Basic, Pro, and Enterprise tiers.",
    "You can upgrade your subscription plan anytime from the dashboard.",
    "Subscription prices vary by region and currency.",
    "Billing cycles run either monthly or annually depending on your plan.",
    "To discontinue service, open Settings > Billing, select your active plan, and confirm termination.",
    "Subscription cancellation requests are reviewed within 24 hours; contact support for cancellation policy details.",
    "The dashboard shows your current plan, next billing date, and subscription status.",
]

vectorstore = FAISS.from_texts(documents, embeddings)

query = "How do I cancel my subscription?"
correct_document = documents[4]
results = vectorstore.similarity_search(query, k=len(documents))

print("BEFORE RE-RANKING (Bi-Encoder / Vector Search)")
print("=" * 70)
for rank, doc in enumerate(results, 1):
    print(f"{rank}. {doc.page_content}")

C:\Users\asus\AppData\Local\Temp\ipykernel_21780\2879246055.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1603.93it/s]


BEFORE RE-RANKING (Bi-Encoder / Vector Search)
1. To stop recurring payments, open Settings > Billing > Cancel Plan and confirm cancellation.
2. You can change your payment method from Settings > Billing without cancelling your plan.
3. You can upgrade your subscription plan anytime from the dashboard.
4. Billing cycles run either monthly or annually depending on your plan.
5. Our subscription plans include Basic, Pro, and Enterprise tiers.
6. The dashboard shows your current plan, next billing date, and subscription status.
7. Subscription prices vary by region and currency.


In [3]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

pairs = [[query, doc.page_content] for doc in results]
scores = reranker.predict(pairs)

reranked = sorted(
    zip(results, scores),
    key=lambda item: float(item[1]),
    reverse=True,
)

print("AFTER RE-RANKING (Cross-Encoder)")
print("=" * 70)
for rank, (doc, score) in enumerate(reranked, 1):
    print(f"{rank}. Cross-encoder score: {float(score):.3f}")
    print(f"   {doc.page_content}")

before_rank = next(
    rank for rank, doc in enumerate(results, 1)
    if doc.page_content == correct_document
)
after_rank = next(
    rank for rank, (doc, _) in enumerate(reranked, 1)
    if doc.page_content == correct_document
)

print("\nRANK COMPARISON")
print("=" * 70)
print(f"Correct cancellation document before reranking: rank {before_rank}")
print(f"Correct cancellation document after reranking : rank {after_rank}")
if after_rank < before_rank:
    print("\nReranking improved the position of the correct document.")
elif after_rank == before_rank:
    print("\nThe document was already correctly ranked by the bi-encoder.")
    print("Reranking confirmed its relevance; a rank change is not guaranteed.")
else:
    print("\nThe correct document moved down; try a larger candidate set.")
print("\nThe bi-encoder finds related billing documents quickly.")
print("The cross-encoder reads the query and each document together")
print("and scores the exact cancellation instruction for relevance.")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2352.88it/s]


AFTER RE-RANKING (Cross-Encoder)
1. Cross-encoder score: 7.584
   To stop recurring payments, open Settings > Billing > Cancel Plan and confirm cancellation.
2. Cross-encoder score: -5.951
   You can change your payment method from Settings > Billing without cancelling your plan.
3. Cross-encoder score: -8.986
   You can upgrade your subscription plan anytime from the dashboard.
4. Cross-encoder score: -10.731
   The dashboard shows your current plan, next billing date, and subscription status.
5. Cross-encoder score: -10.898
   Our subscription plans include Basic, Pro, and Enterprise tiers.
6. Cross-encoder score: -10.982
   Subscription prices vary by region and currency.
7. Cross-encoder score: -11.203
   Billing cycles run either monthly or annually depending on your plan.

RANK COMPARISON
Correct cancellation document before reranking: rank 1
Correct cancellation document after reranking : rank 1

The bi-encoder finds related billing documents quickly.
The cross-encoder reads the